In [149]:
import json
from pathlib import Path
import pandas as pd
import re
import extract as ext
import pdfplumber
import extract_exercises_from_exam_file as parse

In [150]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [151]:
def load_keywords(json_file=ROOT / "json/quimica_keywords.json"):
    with open(json_file) as file:
        return json.load(file)

def load_possible_topics(json_file=ROOT / "json/topics_based_on_exercise_number.json"):
    with open(json_file) as file:
        return json.load(file)

In [152]:
def classify_exercise(statement: str, keywords = None) -> list:
    if keywords is None:
        keywords = load_keywords()

    # convertir saltos de línea en espacios
    statement = re.sub(r"\s+", " " , statement)

    
    # Vocales acentuadas (minúsculas y mayúsculas) → sin acentuar
    tabla = str.maketrans(
        "áéíóúàèìòùäëïöüâêîôûÁÉÍÓÚÀÈÌÒÙÄËÏÖÜÂÊÎÔÛ",
        "aeiouaeiouaeiouaeiouAEIOUAEIOUAEIOUAEIOU"
        )

    statement = statement.translate(tabla)
    topics = {}
    for topic in keywords.keys():
        topic_keywords = keywords[topic]["topic_kw"]
        pattern = re.compile("|".join(topic_keywords), flags=re.IGNORECASE | re.DOTALL)
        keywords_match_count = re.findall(pattern, statement)
        if len(keywords_match_count) > 0:
            topics[topic] = len(keywords_match_count)

    most_occurrences_topic = [k for k,v in topics.items() if v == max(topics.values())]

    return most_occurrences_topic

In [153]:
def possible_topics_based_exercise_number(exercise_number: str, exam_year: int) -> list:

    possible_topics_dict = load_possible_topics()

    # seleccionar modelo de examen en función del año
    if exam_year <= 2019:
        exam_model = "modelo_2010_2019"
        exercise_number_pattern = r"\d{1}"

    elif exam_year <= 2024:
        exam_model = "modelo_2020_2024"
        exercise_number_pattern = r"^([ABC])\d{1}"
    else:
        exam_model = "modelo_2025+"
        exercise_number_pattern = r"^\d{1}?[AB]"

    # temas del ejercicio según número ejercicio y modelo

    # "modelo_2024+": {
    #     "tipo_numeración" : "1A, 1B, 2A, 2B, ..., 4B, 5",
    #     "1": ["configuracion_electronica", ...]
    #     "2": ["equilibrio_quimico", ...],

    topics_for_model_dict = possible_topics_dict[exam_model]

    # obtener el número del ejercicio
    exercise_number_match = re.search(exercise_number_pattern, exercise_number)

    # si se encuentra el número de ejercicio
    if exercise_number_match:

        # extraer la clave del diccionario del modelo de examen

        # 2010 - 2019 - key 1,2,3,...
        if exam_year <= 2019:  
            key = exercise_number_match[0]

            # las keys son: "1", "2,3,4", "5,6"
            if key in "2,3,4":
                key = "2,3,4"
            elif key in "5,6":
                    key = "5,6"
            else:
                key = "1"

        # ejercicios de formulación - NO CLASIFICAR

        # condiciones a cumplir para ser ejercicio de formulación
        formulation_pre_2020 = exam_year <= 2019 and key == "1"
        formulation_2020_2024 =  (2020 <= exam_year <= 2024) and key == "A"

        if formulation_pre_2020 or formulation_2020_2024:
            exercise_topics = ["Formulación"]

        elif exam_year >= 2025 and key == "4A":
            exercise_topics = ["Formulación"]

        else:
            # opciones de tema para el ejercicio suministrado
            exercise_topics = topics_for_model_dict[key]

        return exercise_topics

In [154]:
def classify_exercise_topic(statement: str, topics: list, keywords = None) -> list:
    if keywords is None:
        keywords = load_keywords()

    # convertir saltos de línea en espacios
    statement = re.sub(r"\s+", " " , statement)

    
    # Vocales acentuadas (minúsculas y mayúsculas) → sin acentuar
    tabla = str.maketrans(
        "áéíóúàèìòùäëïöüâêîôûÁÉÍÓÚÀÈÌÒÙÄËÏÖÜÂÊÎÔÛ",
        "aeiouaeiouaeiouaeiouAEIOUAEIOUAEIOUAEIOU"
        )

    statement = statement.translate(tabla)
    # un ejercicio puede ser de diferentes temas
    # elijo el tema por puntuación
    topic_scores = {}
    for topic in topics:
        topic_keywords = keywords[topic]["topic_kw"]
        pattern = re.compile("|".join(topic_keywords), flags=re.IGNORECASE | re.DOTALL)
        kw_matches_count = len(re.findall(pattern, statement))
        if kw_matches_count > 0:
            topic_scores[topic] = kw_matches_count
        
    topics = [k for k,v in topic_scores.items() if v == max(topic_scores.values())]
    return topics

In [155]:
def create_df_exercises_from_exam(exam_file: Path) -> pd.DataFrame:
    exercises_list = []

    # extraer el texto del examen
    exercises_text = parse.extract_exercises_from_pdf(exam_file)

    # clasificar el periodo del examen
    year = int(re.search(r"\d{4}", exam_file.name)[0])
    exam_period = parse.get_exam_period(year)

    # anadir los ejercicios a una lista de diccionarios para
    # luego crear el dataframe
    for ex in parse.split_exercises(exercises_text, exam_period):
        #topic = classify_exercise(statement=ex)
        exercises_list.append({"year" : year, "exam" : exam_file.name, "statement" : ex})
              
    return pd.DataFrame(exercises_list)

In [156]:
def get_exam_option(row):
    if row.year <= 2019:
        exercise_number = row.statement[:21]
        statement = row.statement[22:].strip()
    else:
        exercise_number = row.statement[:2]
        statement = row.statement[3:].strip()
    return exercise_number, statement

In [157]:
def get_convocatoria(exam):
    if exam.startswith("química") or exam.startswith("quimica"):
        return exam.split(" - ")[-1].replace(".pdf", "").title()
    else:
        return exam

Clasificar ejercicios de un archivo

In [158]:
file = ROOT / "examenes_quimica/2014_quimica/química - 2014 - reserva 2.pdf"

In [159]:
df = create_df_exercises_from_exam(file)

In [160]:
df["exercise_number"], df["statement"] = zip(*df.apply(get_exam_option, axis=1))
df["convocatoria"] = df.exam.apply(get_convocatoria)

In [161]:
df.head()

,year,exam,statement,exercise_number,convocatoria
0,2014,química - 2014 - reserva 2.pdf,Formule o nombre los siguientes compuestos: a)...,"Ejercicio 1, opción A",Reserva 2
1,2014,química - 2014 - reserva 2.pdf,Razone si los siguientes enunciados son verdad...,"Ejercicio 2, opción A",Reserva 2
2,2014,química - 2014 - reserva 2.pdf,a) Escriba la ecuación de equilibrio de solubi...,"Ejercicio 3, opción A",Reserva 2
3,2014,química - 2014 - reserva 2.pdf,a) Razone si las reacciones con valores positi...,"Ejercicio 4, opción A",Reserva 2
4,2014,química - 2014 - reserva 2.pdf,El ácido nítrico reacciona con el sulfuro de h...,"Ejercicio 5, opción A",Reserva 2


In [172]:
exercise_topics = []
matches = []
for idx, row in df.iterrows():
    possible_topics = possible_topics_based_exercise_number(row.exercise_number, row.year)
    
    if possible_topics :
        # formulación
        if possible_topics[0] == "Formulación":
            exercise_topics.append(["Formulación"])
        else:
            ex_topics = []
        
            topics = classify_exercise_topic(row.statement, possible_topics)
            if topics:
                exercise_topics.append(topics)

df["topic"] = exercise_topics
df = df.explode("topic")


In [167]:
print(*df.statement, sep="\n\n")

Formule o nombre los siguientes compuestos: a) Hidróxido de estroncio b) Dicromato de bario
c) Ácido 2,3-dihidroxibutanoico d) Al₂O₃ e) H₂MnO₄ f) CH₃CH₂CONH₂.

Razone si los siguientes enunciados son verdaderos o falsos:
a) Los compuestos covalentes conducen la corriente eléctrica.
b) Todos los compuestos covalentes tienen puntos de fusión elevados.
c) Todos los compuestos iónicos, disueltos en agua, son buenos conductores de la electricidad.

a) Escriba la ecuación de equilibrio de solubilidad en agua del Al(OH)₃.
b) Escriba la relación entre solubilidad y Ks para el Al(OH)₃.
c) Razone cómo afecta a la solubilidad del Al(OH)₃ un aumento del pH.

a) Razone si las reacciones con valores positivos de ΔSº siempre son espontáneas a alta
temperatura.
b) La siguiente reacción (sin ajustar) es exotérmica: C₃H₈O (l) + O₂ (g) → CO₂ (g) + H₂O (g).
Justifique si a presión constante se desprende más, igual o menos calor que a volumen constante.
c) Razone si en un proceso exotérmico la entalpía de 

In [143]:
for i in df[df.topic == "redox"].statement:
    print(f"{i}\n\n")

El ácido nítrico reacciona con el sulfuro de hidrógeno dando azufre elemental (S), monóxido de
nitrógeno y agua.
a) Escriba y ajuste por el método del ion-electrón la reacción molecular correspondiente.
b) Determine el volumen de sulfuro de hidrógeno, medido a 60ºC y 1 atm, necesario para que
reaccione con 500 mL de ácido nítrico 0,2 M.
Dato: R = 0,082 atm·L·moI⁻¹·K⁻¹.


El ácido nítrico reacciona con el sulfuro de hidrógeno dando azufre elemental (S), monóxido de
nitrógeno y agua.
a) Escriba y ajuste por el método del ion-electrón la reacción molecular correspondiente.
b) Determine el volumen de sulfuro de hidrógeno, medido a 60ºC y 1 atm, necesario para que
reaccione con 500 mL de ácido nítrico 0,2 M.
Dato: R = 0,082 atm·L·moI⁻¹·K⁻¹.




In [173]:
df

,year,exam,statement,exercise_number,convocatoria,topic
0,2014,química - 2014 - reserva 2.pdf,Formule o nombre los siguientes compuestos: a)...,"Ejercicio 1, opción A",Reserva 2,Formulación
1,2014,química - 2014 - reserva 2.pdf,Razone si los siguientes enunciados son verdad...,"Ejercicio 2, opción A",Reserva 2,enlace_quimico
2,2014,química - 2014 - reserva 2.pdf,a) Escriba la ecuación de equilibrio de solubi...,"Ejercicio 3, opción A",Reserva 2,solubilidad
3,2014,química - 2014 - reserva 2.pdf,a) Razone si las reacciones con valores positi...,"Ejercicio 4, opción A",Reserva 2,termoquimica
4,2014,química - 2014 - reserva 2.pdf,El ácido nítrico reacciona con el sulfuro de h...,"Ejercicio 5, opción A",Reserva 2,acido_base
5,2014,química - 2014 - reserva 2.pdf,Una disolución acuosa 10⁻²M de ácido benzoico ...,"Ejercicio 6, opción B",Reserva 2,acido_base
6,2014,química - 2014 - reserva 2.pdf,Formule o nombre los siguientes compuestos: a)...,"Ejercicio 1, opción A",Reserva 2,Formulación
7,2014,química - 2014 - reserva 2.pdf,Un recipiente de 1 litro de capacidad está lle...,"Ejercicio 2, opción A",Reserva 2,cantidad_quimica
8,2014,química - 2014 - reserva 2.pdf,Escriba la configuración electrónica de:\na) U...,"Ejercicio 3, opción A",Reserva 2,configuracion_electronica
9,2014,química - 2014 - reserva 2.pdf,Responda razonadamente:\na) En una disolución ...,"Ejercicio 4, opción A",Reserva 2,acido_base


In [ ]:
exam_files = list((ROOT / "examenes_quimica").rglob("*.pdf"))

In [ ]:
# df_clasificados = pd.DataFrame()
# for file in exam_files:
#     df = create_df_exercises_from_exam(file).explode("topic")
#     df_clasificados = pd.concat([df_clasificados, df], axis=0)

# df = df_clasificados

In [ ]:
# df.to_csv("examenes_oficiales_clasificados.csv", index = False)

In [37]:
df = pd.read_csv("examenes_oficiales_clasificados.csv")

In [38]:
df.sample(5)

,year,exam,topic,statement,exercise_number,convocatoria
350,2022,química - 2022 - julio.pdf,reactividad_organica,a) Escriba dos compuestos isómeros de fórmula ...,B6,Julio
635,2025,2025_quimica_Quimica - Examen_Suplente2-B_Quim...,formulacion,Nombre o formule los siguientes compuestos:\na...,4A,2025_quimica_Quimica - Examen_Suplente2-B_Quim...
945,2023,química - 2023 - reserva 2.pdf,reactividad_organica,Considere los siguientes tipos de compuestos o...,B4,Reserva 2
564,2010,química - 2010 - reserva 2.pdf,configuracion_electronica,a) Justifique cómo es el tamaño de un átomo co...,"Ejercicio 2, opción A",Reserva 2
868,2014,química - 2014 - reserva 3.pdf,redox,"Se hace pasar durante 2,5 horas una corriente ...","Ejercicio 6, opción B",Reserva 3


In [39]:
df.isna().sum()

year               0
exam               0
topic              6
statement          0
exercise_number    0
convocatoria       0
dtype: int64

In [40]:
df[df.topic.isna()]

,year,exam,topic,statement,exercise_number,convocatoria
149,2016,química - 2016 - reserva 3.pdf,NaN,En un matraz cerrado de 5 L hay 42 g de N₂ a 2...,"Ejercicio 2, opción A",Reserva 3
414,2018,química - 2018 - septiembre.pdf,NaN,"Dados los siguientes reactivos HI, I₂, H₂/cata...","Ejercicio 4, opción A",Septiembre
534,2012,quimica - 2012 - reserva 1.pdf,NaN,Se preparan 25 mL de una disolución 2’5M de Fe...,"Ejercicio 5, opción A",Reserva 1
610,2010,química - 2010 - reserva 1.pdf,NaN,Indique los reactivos adecuados para realizar ...,"Ejercicio 4, opción A",Reserva 1
715,2024,química - 2024 - julio.pdf,NaN,Escriba las siguientes reacciones:\na) CH₃CH=C...,B6,Julio
836,2014,química - 2014 - reserva 4.pdf,NaN,Justifique la veracidad o falsedad de las sigu...,"Ejercicio 2, opción A",Reserva 4


In [41]:
df.topic.value_counts().sort_values(ascending=False)

topic
acido-base                   219
formulacion                  145
redox                        119
equilibrio                   106
configuracion_electronica     88
reactividad_organica          84
enlace_quimico                82
solubilidad                   70
termoquimica                  45
propiedades_periodicas        28
cantidad_quimica              25
cinetica                      18
Name: count, dtype: int64